# 🧬 GNINA ile Docking (Google Colab · ücretsiz T4 GPU)

Bu notebook, Remedia pipeline'ının **docking** adımını senin yerine yapar.
Docking motoru artık **GNINA** (GPU'da çalışan derin öğrenmeli docking) —
AutoDock Vina tamamen bırakıldı. Colab'ın **ücretsiz T4 GPU**'sunda çalışır;
sonuçta inen `docking_scores.csv` dosyasını Remedia'ya yükleyip normal
pipeline'a (ADMET → sıralama → dashboard) hiçbir şey değiştirmeden devam edersin.

### 📋 Nasıl kullanılır — SADECE ŞU 3 KURAL:
1. Üstten **Runtime ▸ Change runtime type ▸ Hardware accelerator ▸ GPU (T4)** seç.
2. Hücreleri **yukarıdan aşağıya SIRAYLA** çalıştır (her hücrede `Shift+Enter`).
3. Bir sonraki hücreye geçmeden önce, o hücrenin çıktısında **`✅`** işaretini gör.
   `✅` yoksa o adımı düzeltmeden ilerleme.

> Her adımın üstünde ne yaptığını, ne kadar süreceğini ve devam etmeden önce
> neyi görmen gerektiğini yazan **mavi bir not** var. Tahmin etmene gerek yok.


## 🔵 ADIM 1: GPU kontrolü
**Ne yapıyor:** Colab'ın sana ücretsiz bir GPU (Tesla T4) verip vermediğini kontrol eder.
**Ne kadar sürer:** ~5 saniye.
**Devam etmeden önce:** Çıktıda `✅ GPU HAZIR` yazısını gör. `⚠️` görürsen, notta yazan menü adımını yap ve bu hücreyi TEKRAR çalıştır.


In [ ]:
import subprocess

# nvidia-smi GPU'yu listeler; GPU yoksa hata verir. "Tesla T4" arıyoruz.
proc = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(proc.stdout or proc.stderr)

if proc.returncode != 0 or "NVIDIA-SMI" not in (proc.stdout or ""):
    print("=" * 70)
    print("⚠️ GPU bulunamadı!")
    print("Üstteki menüden:  Runtime > Change runtime type >")
    print("                  Hardware accelerator > GPU (T4) seç,")
    print("sonra BU HÜCREYİ TEKRAR çalıştır.")
    print("=" * 70)
else:
    if "T4" not in proc.stdout:
        print("ℹ️ Not: Beklenen Tesla T4 değil ama bir GPU bulundu — devam edebilirsin.")
    print("✅ GPU HAZIR — ADIM 2'ye geç.")

## 🔵 ADIM 2: GNINA kurulumu + Remedia reposunun klonlanması
**Ne yapıyor:** GNINA'nın GPU destekli resmi binary'sini indirir, Remedia repo'nu klonlar ve GNINA'nın gerçekten çalışıp CNN/GPU skorlama seçeneklerine sahip olduğunu doğrular.
**Ne kadar sürer:** ~2–4 dakika (GNINA binary'si büyüktür, bir kere iner).
**Devam etmeden önce:** En sonda `✅ Kurulum tamam, ADIM 3'e geç` yazısını gör. Hata görürsen README'deki "Sorun mu yaşıyorsun?" bölümüne bak.


In [ ]:
import os, sys, stat, subprocess, urllib.request

# --- 1) Remedia reposunu klonla (zaten varsa tekrar klonlamaz) -------------
if not os.path.isdir("Remedia"):
    subprocess.run(["git", "clone", "-q",
                    "https://github.com/mehmetg06/Remedia.git"], check=True)
print("• Remedia reposu hazır (Remedia/)")

# --- 2) RDKit (ligandları 3D'ye hazırlamak için) ---------------------------
try:
    import rdkit  # noqa: F401
    print("• RDKit zaten kurulu")
except ImportError:
    print("• RDKit kuruluyor...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rdkit"], check=True)

# --- 3) GNINA GPU binary'sini indir ----------------------------------------
# GNINA'nın resmi statik binary'si CUDA'yı doğrudan kullanır; Colab'ın T4'ünde
# ayrıca derleme yapmadan GPU'da çalışır. En güncel sürümü dener, olmazsa
# bilinen kararlı sürümlere düşer.
GNINA_PATH = "/usr/local/bin/gnina"
GNINA_URLS = [
    "https://github.com/gnina/gnina/releases/download/v1.3/gnina",
    "https://github.com/gnina/gnina/releases/download/v1.1/gnina",
    "https://github.com/gnina/gnina/releases/download/v1.0.3/gnina",
]

if os.path.exists(GNINA_PATH) and os.path.getsize(GNINA_PATH) > 1_000_000:
    print("• GNINA zaten indirilmiş")
else:
    # Bazı statik binary'lerin ihtiyaç duyabileceği paylaşımlı kütüphaneler (güvenlik ağı)
    subprocess.run("apt-get -qq install -y libboost-all-dev >/dev/null 2>&1", shell=True)
    indirildi = False
    for url in GNINA_URLS:
        try:
            print(f"• GNINA indiriliyor: {url}")
            urllib.request.urlretrieve(url, GNINA_PATH)
            if os.path.getsize(GNINA_PATH) > 1_000_000:
                indirildi = True
                break
        except Exception as e:
            print(f"  (bu sürüm alınamadı: {e})")
    if not indirildi:
        raise RuntimeError(
            "GNINA binary'si indirilemedi. Geçici bir erişim sorunu olabilir — "
            "bu hücreyi tekrar çalıştır. Sürerse README 'Sorun mu yaşıyorsun?' bölümüne bak."
        )
    mode = os.stat(GNINA_PATH).st_mode
    os.chmod(GNINA_PATH, mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

# --- 4) GNINA gerçekten çalışıyor mu + CNN/GPU desteği var mı? --------------
ver = subprocess.run([GNINA_PATH, "--version"], capture_output=True, text=True)
print("\n--- gnina --version ---")
print(((ver.stdout or "") + (ver.stderr or "")).strip()[:400])

helptxt = subprocess.run([GNINA_PATH, "--help"], capture_output=True, text=True)
h = ((helptxt.stdout or "") + (helptxt.stderr or "")).lower()
cnn_gpu = ("cnn_scoring" in h) or ("gpu" in h) or ("--device" in h)
if cnn_gpu:
    print("• GNINA çalışıyor ve CNN/GPU skorlama seçenekleri mevcut ✔")
    print("  (ADIM 4'teki gerçek docking GPU'da CNN rescoring ile yapılacak.)")
else:
    print("⚠️ GNINA çalıştı ama yardım çıktısında CNN/GPU referansı görülemedi — "
          "yine de ADIM 4 gerçek docking'te GPU kullanımını doğrulayacak.")

print("\n" + "=" * 70)
print("✅ Kurulum tamam, ADIM 3'e geç")
print("=" * 70)

## 🔵 ADIM 3: Girdi seçimi — reseptör, pocket ve moleküller
**Ne yapıyor:** Hangi reseptörü (protein), hangi pocket merkezini/kutu boyutunu ve hangi molekülleri (ligand) dockladığını **kod yazmadan** kutucuklardan seçmeni sağlar; sonra kaç molekül ve hangi isimlerle işleneceğini ekrana yazar.
**Ne kadar sürer:** ~10–30 saniye (reseptör indirmesi dahil).
**Devam etmeden önce:** Alttaki listede molekül isimlerini gözünle oku — "evet, bunlar doğru moleküller" dediğinde ADIM 4'e geç. Değilse kutucukları düzelt ve hücreyi tekrar çalıştır.

> **Kutucuklar ne işe yarar?**
> - **UNIPROT_ID:** Reseptörün AlphaFold DB kimliği. Varsayılan `P30405` (CypD).
> - **RECEPTOR_PATH:** Elinde hazır `.pdb`/`.pdbqt` varsa repo içi yolunu yaz (ör. `Remedia/data/P30405_alphafold.pdb`). Boş bırakırsan UNIPROT_ID'den otomatik indirilir.
> - **CENTER_X/Y/Z:** Pocket (bağlanma cebi) merkezinin koordinatları — Remedia `config.yaml`'daki `pocket_center` ile aynı değerleri kullan.
> - **SIZE_X/Y/Z:** Docking kutusu boyutu (varsayılan 20 20 20).
> - **LIGAND_SMI_FILE:** Molekül listesi `.smi` dosyası. Remedia UI'da ürettiysen `Remedia/data/generated.smi`.
> - **MANUAL_SMILES:** Dosya yerine elle SMILES yapıştırmak istersen buraya yaz (her satır: `SMILES isim`). Doluysa dosyayı değil bunu kullanır.


In [ ]:
#@title ADIM 3 ayarları — kutucukları doldur, sonra bu hücreyi çalıştır { display-mode: "form" }
#@markdown ### 🧫 Reseptör (protein)
UNIPROT_ID = "P30405"  #@param {type:"string"}
RECEPTOR_PATH = ""  #@param {type:"string"}
#@markdown ### 📦 Pocket merkezi (Å) — config.yaml'daki pocket_center ile aynı olsun
CENTER_X = 5.00  #@param {type:"number"}
CENTER_Y = -1.02  #@param {type:"number"}
CENTER_Z = -15.56  #@param {type:"number"}
#@markdown ### 📦 Docking kutusu boyutu (Å)
SIZE_X = 20  #@param {type:"number"}
SIZE_Y = 20  #@param {type:"number"}
SIZE_Z = 20  #@param {type:"number"}
#@markdown ### 🧪 Ligandlar (moleküller)
LIGAND_SMI_FILE = "Remedia/data/generated.smi"  #@param {type:"string"}
#@markdown Dosya yoksa aşağıya elle SMILES yapıştır (her satır: `SMILES isim`). Doluysa üstteki dosya yerine bu kullanılır.
MANUAL_SMILES = ""  #@param {type:"string"}
MAX_MOLECULES = 50  #@param {type:"integer"}

import urllib.request
from pathlib import Path

# --- Reseptörü hazırla (verilen dosya yoksa AlphaFold'dan indir) -----------
def _resolve_receptor():
    if RECEPTOR_PATH.strip() and Path(RECEPTOR_PATH).exists():
        return str(Path(RECEPTOR_PATH).resolve())
    if RECEPTOR_PATH.strip():
        print(f"ℹ️ Verilen RECEPTOR_PATH bulunamadı ({RECEPTOR_PATH}); "
              f"UNIPROT_ID={UNIPROT_ID} üzerinden indiriliyor.")
    dest = Path(f"receptor_{UNIPROT_ID}.pdb").resolve()
    if not dest.exists():
        url = f"https://alphafold.ebi.ac.uk/files/AF-{UNIPROT_ID}-F1-model_v4.pdb"
        print(f"• Reseptör indiriliyor (AlphaFold {UNIPROT_ID})...")
        urllib.request.urlretrieve(url, str(dest))
    return str(dest)

RECEPTOR = _resolve_receptor()
print(f"• Reseptör: {RECEPTOR}")

# --- Molekülleri oku: [(isim, smiles), ...] --------------------------------
def parse_smi_text(text):
    mols = []
    for line in text.splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        parts = line.split()
        smiles = parts[0]
        name = parts[1] if len(parts) > 1 else f"mol_{len(mols) + 1}"
        mols.append((name, smiles))
    return mols

if MANUAL_SMILES.strip():
    molecules = parse_smi_text(MANUAL_SMILES)
    print("• Kaynak: elle girilen SMILES")
elif Path(LIGAND_SMI_FILE).exists():
    molecules = parse_smi_text(Path(LIGAND_SMI_FILE).read_text())
    print(f"• Kaynak: {LIGAND_SMI_FILE}")
else:
    raise FileNotFoundError(
        f"Ligand dosyası bulunamadı: {LIGAND_SMI_FILE}\n"
        "Ya doğru yolu yaz (ör. Remedia/data/ligands_example.smi) ya da "
        "MANUAL_SMILES kutusuna elle SMILES yapıştır, sonra hücreyi tekrar çalıştır."
    )

molecules = molecules[:MAX_MOLECULES]
CENTER = (CENTER_X, CENTER_Y, CENTER_Z)
SIZE = (SIZE_X, SIZE_Y, SIZE_Z)

# --- Görsel onay için tabloyu yazdır ---------------------------------------
print()
print("=" * 70)
print(f"İŞLENECEK MOLEKÜLLER  ({len(molecules)} adet)")
print("-" * 70)
print(f"{'#':>2}  {'İSİM':<24} SMILES")
print("-" * 70)
for i, (name, smiles) in enumerate(molecules, 1):
    print(f"{i:>2}  {name:<24} {smiles}")
print("=" * 70)
if not molecules:
    print("⚠️ Hiç molekül seçilmedi! LIGAND_SMI_FILE ya da MANUAL_SMILES'ı düzelt.")
else:
    print(f"📦 Pocket merkezi: {CENTER}   ·   Kutu boyutu: {SIZE}")
    print("👀 Yukarıdaki isimler DOĞRU moleküllerse ADIM 4'e geç.")

## 🔵 ADIM 4: GNINA ile toplu docking (GPU)
**Ne yapıyor:** Her molekülü tek tek RDKit ile 3D'ye hazırlar (ligand_prep.py ile aynı mantık), sonra GNINA'yı **GPU'da CNN rescoring** ile çalıştırıp bağlanma enerjisini (affinity, kcal/mol) hesaplar.
**Ne kadar sürer:** Molekül başına ~15–60 saniye (T4'te). İlerleme çubuğu tahmini kalan süreyi gösterir.
**Devam etmeden önce:** İlerleme çubuğu dolup her molekül için `✅ [isim]: affinity=X kcal/mol` satırını gör. Bir molekül `❌` verirse sorun değil — diğerleri devam eder, o satır boş skorla raporlanır.

> **Not:** GNINA'nın **CNN rescoring** adımı GPU'da çalışır; skorların üretilmesi GNINA'nın GPU'yu gerçekten kullandığının kanıtıdır. `affinity` değeri daha negatifse bağlanma daha güçlüdür.


In [ ]:
import re, time, subprocess
from pathlib import Path
from rdkit import Chem
from rdkit.Chem import AllChem
from tqdm.auto import tqdm

GNINA_PATH = "/usr/local/bin/gnina"
OUT_DIR = Path("gnina_out")
OUT_DIR.mkdir(exist_ok=True)


def prepare_ligand_sdf(smiles, name):
    """ligand_prep.py ile AYNI mantık: RDKit ile 3D konformasyon üret + MMFF ile
    optimize et, SDF olarak yaz. GNINA SDF'i doğrudan okur; Vina'daki ayrı
    PDBQT dönüşüm adımına gerek yoktur."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    mol = Chem.AddHs(mol)
    if AllChem.EmbedMolecule(mol, randomSeed=42) != 0:
        return None
    try:
        AllChem.MMFFOptimizeMolecule(mol)
    except Exception:
        pass
    sdf = OUT_DIR / f"{name}.sdf"
    w = Chem.SDWriter(str(sdf))
    w.write(mol)
    w.close()
    return sdf


def parse_affinity(out_sdf, stdout):
    """En iyi (1. sıra) pozun affinity'sini kcal/mol olarak döndürür.
    Önce çıktı SDF'indeki property'lere, olmazsa stdout tablosuna bakar."""
    try:
        supp = Chem.SDMolSupplier(str(out_sdf), removeHs=False)
        for m in supp:
            if m is None:
                continue
            for key in ("minimizedAffinity", "CNNaffinity", "affinity"):
                if m.HasProp(key):
                    return float(m.GetProp(key))
            break
    except Exception:
        pass
    for line in stdout.splitlines():
        m = re.match(r"\s*1\s+(-?\d+\.\d+)", line)
        if m:
            return float(m.group(1))
    return None


results_rows = []
n = len(molecules)
print(f"Toplam {n} molekül GNINA (GPU · CNN rescoring) ile docklanacak.\n")

start = time.time()
bar = tqdm(molecules, desc="GNINA", unit="mol")
for idx, (name, smiles) in enumerate(bar, 1):
    sdf = prepare_ligand_sdf(smiles, name)
    if sdf is None:
        print(f"❌ {name}: geçersiz SMILES / 3D üretilemedi — atlandı")
        results_rows.append({"ligand": name, "affinity_kcal_mol": None})
        continue

    out_sdf = OUT_DIR / f"{name}_docked.sdf"
    cmd = [
        GNINA_PATH,
        "-r", RECEPTOR,
        "-l", str(sdf),
        "--center_x", str(CENTER[0]), "--center_y", str(CENTER[1]), "--center_z", str(CENTER[2]),
        "--size_x", str(SIZE[0]), "--size_y", str(SIZE[1]), "--size_z", str(SIZE[2]),
        "--cnn_scoring", "rescore",
        "--seed", "42",
        "-o", str(out_sdf),
    ]
    proc = subprocess.run(cmd, capture_output=True, text=True)
    aff = parse_affinity(out_sdf, proc.stdout)

    if aff is not None:
        print(f"✅ {name}: affinity={aff:.2f} kcal/mol")
    else:
        son = (proc.stderr or proc.stdout or "").strip().splitlines()
        neden = " | ".join(son[-2:]) if son else "bilinmeyen hata"
        print(f"❌ {name}: docking başarısız — {neden}")
    results_rows.append({"ligand": name, "affinity_kcal_mol": aff})

    # tqdm çubuğuna "X/Y · tahmini kalan: Z dk" bilgisini yaz
    gecen = time.time() - start
    kalan_dk = (gecen / idx) * (n - idx) / 60.0
    bar.set_postfix_str(f"{idx}/{n} · tahmini kalan: {kalan_dk:.1f} dk")

basarili = sum(1 for r in results_rows if r["affinity_kcal_mol"] is not None)
print(f"\n✅ GNINA bitti: {basarili}/{n} molekül başarıyla skorlandı. ADIM 5'e geç.")

## 🔵 ADIM 5: Sonuçları indir
**Ne yapıyor:** Tüm skorları, Remedia'nın beklediği **birebir aynı** formatta (`ligand,affinity_kcal_mol`) tek bir `docking_scores.csv` dosyasına yazar ve **otomatik olarak bilgisayarına indirir**.
**Ne kadar sürer:** ~5 saniye.
**Devam etmeden önce:** Tarayıcın `docking_scores.csv` dosyasını indirdiğinde işin bitti. Alttaki büyük yeşil mesajdaki talimatı izle.

> Çıktıda **BAŞKA HİÇBİR EK SÜTUN YOKTUR** — sadece `ligand` ve `affinity_kcal_mol`. Bu sayede mevcut `admet_filter.py` ve `rank_report.py` dosyayı hiç değiştirmeden okuyabilir.


In [ ]:
import csv
from google.colab import files

RUN_CSV = "docking_scores.csv"

# Mevcut docking_scores.csv formatının BİREBİR aynısı: sadece iki sütun.
with open(RUN_CSV, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["ligand", "affinity_kcal_mol"])   # BAŞKA SÜTUN YOK
    for r in results_rows:
        aff = r["affinity_kcal_mol"]
        writer.writerow([r["ligand"], "" if aff is None else round(aff, 4)])

print(f"'{RUN_CSV}' yazıldı: {len(results_rows)} satır · sütunlar = ligand, affinity_kcal_mol")
print("İndiriliyor...")
files.download(RUN_CSV)  # tarayıcı otomatik indirir

print()
print("#" * 72)
print("#" + " " * 70 + "#")
print("#" + "🎉  BİTTİ!".center(69) + "#")
print("#" + " " * 70 + "#")
print("#" + "İndirilen  docking_scores.csv  dosyasını".center(70) + "#")
print("#" + "Remedia/results/<run_id>/  klasörüne".center(70) + "#")
print("#" + "'docking_scores.csv' olarak YÜKLE (üzerine yaz).".center(70) + "#")
print("#" + " " * 70 + "#")
print("#" + "Sonra Codespaces'te normal pipeline'ı devam ettir:".center(70) + "#")
print("#" + "ADMET → sıralama → dashboard.".center(70) + "#")
print("#" + "Yeni bir script çalıştırmana GEREK YOK — format birebir aynı.".center(70) + "#")
print("#" + " " * 70 + "#")
print("#" * 72)